# Cargar las librerias

In [19]:
import tensorflow as tf

from tensorflow.keras import layers, models
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cargar los datos

In [20]:
val_glove = np.load('../datos/encuestav1_gloVe_Values.npy', allow_pickle=True)
val_glove.shape

(460,)

In [21]:
data = pd.read_csv('../datos/train_poll_v1s1.csv')
labels = data['ai']
labels = np.array(labels)
labels.shape

(460,)

In [22]:
# Split data into training and validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(val_glove, labels, test_size=0.2, random_state=42)


# Crear el modelo y entrenar

In [23]:


class TransformerEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        attn_output = self.att(inputs, inputs, attention_mask=mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Main model
embedding_dim = 300
num_heads = 2
ff_dim = 128

# Input layer allows variable sequence length
inputs = tf.keras.Input(shape=(None, embedding_dim), name="inputs")

# Optional mask for padding (zeros = pad)
mask_input = tf.keras.Input(shape=(None,), dtype=tf.bool, name="mask")

# Transformer with attention mask
x = TransformerEncoderLayer(embedding_dim, num_heads, ff_dim)(inputs, mask=mask_input)

# Output
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(1, activation='sigmoid')(x)

# Model
model = tf.keras.Model(inputs=[inputs, mask_input], outputs=x)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, None, 300) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask (InputLayer)   │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 300) │    800,528 │ inputs[0][0],     │
│ (TransformerEncode… │                   │            │ mask[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │     19,264 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         65 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 819,857 (3.13 MB)

 Trainable params: 819,857 (3.13 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Mask creation (to handle padding for variable length sequences)
mask_train = np.any(X_train != 0, axis=-1)
mask_val = np.any(X_val != 0, axis=-1)
mask_val

C:\Users\angel\AppData\Local\Temp\ipykernel_8308\4047965678.py:2: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mask_train = np.any(X_train != 0, axis=-1)
C:\Users\angel\AppData\Local\Temp\ipykernel_8308\4047965678.py:3: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mask_val = np.any(X_val != 0, axis=-1)


True

In [ ]:

# Train the model
model.fit([X_train, mask_train], y_train, batch_size=32, epochs=5, validation_data=([X_val, mask_val], y_val))

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type numpy.ndarray).

In [33]:
import numpy as np

# Example of sequences (4 tokens per sequence)
X_train = [
    [0.5, 0.3, 0, 0],  # sequence 1 with padding at 2 and 3
    [0.1, 0.4, 0.2, 0]  # sequence 2 with padding at 3
]

# Create a mask where True = valid tokens, False = padding tokens
mask_train = X_train != 0  # Check for non-zero values
print(mask_train)


True
